In [13]:
import pandas as pd # Librería de lectura de datos
import numpy as np # Librería de cálculo numérico

import matplotlib.pyplot as plt # Librería de visualización de datos

from sklearn.model_selection import train_test_split,GridSearchCV # Función para dividir los datos en entrenamiento y prueba
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder,LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier,plot_tree, export_text

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve # Funciones para evaluar el rendimiento del modelo

from sklearn.feature_selection import (VarianceThreshold, SelectKBest,
                                      f_classif, RFE, SelectFromModel)
from sklearn import svm
from sklearn.metrics import confusion_matrix as cm_func
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix as cm_func2
from xgboost import XGBClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.metrics import confusion_matrix as cm_func3

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [14]:
# Parámetros
INPUT = "../clean_data/mushrooms_limpio.parquet"
OUTPUT_PARQUET = "../clean_data/mushrooms_limpio_imputed.parquet"
OUTPUT_CSV = "../clean_data/mushrooms_limpio_imputed.csv"
N_NEIGHBORS = 5

# Resumen del análisis
- **Forma:** (8124, 21)
- **Todas las columnas son categóricas** (tipo `str`).
- **Nulos:** 0 en todas las columnas.
- **Balance de clases:** `e` 51.8%, `p` 48.2% (casi balanceado).
- **Variabilidad:** varias columnas con baja cardinalidad (2–6 valores); algunas con más (hasta 12).

Recomendaciones rápidas:
- Para clustering con algoritmos basados en distancia (KMeans, KNN): convertir categóricas con `OneHotEncoder` (sparse), luego reducir dimensión (TruncatedSVD o UMAP) y aplicar `StandardScaler`.
- Si quieres evitar expansión dimensional: usar codificaciones por frecuencia / target encoding / hashing, o usar `KModes` / `k-prototypes` para datos categóricos.
- No escalar variables para métodos basados en árboles o `KModes`.
- Imputación: `SimpleImputer(strategy='most_frequent')` si aparecen nulos en otras fuentes.

He incluido a continuación un ejemplo de pipeline reproducible que puedes ejecutar (celda 3).

In [15]:
# 1) Cargar datos
df = pd.read_parquet(INPUT)

# 2) Reemplazar '?' por NaN en la columna 'stalk root' (y opcionalmente en todo el DF si hay más '?')
df = df.replace('?', np.nan)

# 3) Identificar columnas categóricas
cat_cols = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

# 4) Guardar mapeos de categorías y convertir categóricas a códigos (preservando NaN como np.nan)
mappings = {}
df_num = df.copy()
for col in cat_cols:
    df_num[col] = df_num[col].astype('category')
    mappings[col] = df_num[col].cat.categories.tolist()  # lista de categorías en orden
    codes = df_num[col].cat.codes.replace(-1, np.nan).astype('float')  # -1 -> NaN
    df_num[col] = codes

# 5) Aplicar KNNImputer sobre la matriz numérica
imputer = KNNImputer(n_neighbors=N_NEIGHBORS, weights='uniform', metric='nan_euclidean')
imputed_array = imputer.fit_transform(df_num)
df_imputed = pd.DataFrame(imputed_array, columns=df_num.columns, index=df.index)

# 6) Decodificar las columnas categóricas: redondear los códigos y mapear a categorías
for col in cat_cols:
    cats = mappings[col]
    # redondear y convertir a int; clip para evitar índices fuera de rango
    codes = df_imputed[col].round().astype('Int64')  # permite valores NA
    # Mapear códigos NA a NA; para valores numéricos, asegurarnos que estén en rango
    def code_to_cat(val):
        if pd.isna(val):
            return np.nan
        i = int(val)
        i = max(0, min(i, len(cats) - 1))
        return cats[i]
    df_imputed[col] = codes.apply(code_to_cat).astype('category')

# 7) Opcional: restaurar tipos numéricos originales (si quieres)
# Si tenías columnas enteras que ahora son float por el imputer, conviértelas aquí.
# Ejemplo (descomentar y ajustar):
# int_cols = ['some_int_col']
# for c in int_cols:
#     df_imputed[c] = df_imputed[c].round().astype('Int64')

# 8) Guardar resultados
df_imputed.to_parquet(OUTPUT_PARQUET, index=False)
df_imputed.to_csv(OUTPUT_CSV, index=False)

print("Imputación completada. Archivos guardados:", OUTPUT_PARQUET, OUTPUT_CSV)

Imputación completada. Archivos guardados: ../clean_data/mushrooms_limpio_imputed.parquet ../clean_data/mushrooms_limpio_imputed.csv


In [16]:
cat_cols = df.select_dtypes(include=['object','string','category']).columns.tolist()

preproc = ColumnTransformer([
        ('imp_ohe', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
    ]), cat_cols),
], remainder='drop', sparse_threshold=0)

# Ajusta n_components según dimensionalidad (aquí 30 como ejemplo)
n_components = min(30, max(2, len(cat_cols)))
pipe = Pipeline([
    ('preproc', preproc),
    ('svd', TruncatedSVD(n_components=n_components, random_state=0)),
    ('scaler', StandardScaler()),
    ('km', KMeans(n_clusters=3, random_state=0))
])

pipe.fit(df)
labels = pipe.named_steps['km'].labels_
print('clusters assigned:', len(set(labels)))

# Guarda etiquetas
df['cluster'] = labels
df.to_csv('../clean_data/mushrooms_with_clusters.csv', index=False)
print('Wrote ../clean_data/mushrooms_with_clusters.csv')

clusters assigned: 3
Wrote ../clean_data/mushrooms_with_clusters.csv


## Limpieza avanzada
En esta celda aplicamos: eliminación de duplicados, detección/agrupación de niveles raros en categóricas, imputación (numérica: mediana, categórica: moda) y detección básica de outliers numéricos (IQR).
- Para variables categóricas: se agrupan niveles con frecuencia < 1% en `rare` y luego se imputan con la categoría más frecuente.
- Para variables numéricas: se detectan outliers por IQR (se pueden eliminar o winsorizar manualmente).
- Finalmente se guarda `clean_data/mushrooms_cleaned_advanced.csv`.

In [17]:
# 1) Duplicados
dups = df.duplicated().sum()
print('duplicates:', dups)
df = df.drop_duplicates()
print('After drop duplicates shape:', df.shape)

# 2) Identificar numéricas / categóricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object','string','category']).columns.tolist()
print('num_cols:', num_cols)
print('cat_cols len:', len(cat_cols))

# Convertir categóricas 'category' a object para poder añadir niveles nuevos
for c in cat_cols:
    # evite Pandas4Warning usando isinstance sobre el dtype CategoricalDtype
    if isinstance(df[c].dtype, pd.CategoricalDtype):
        df[c] = df[c].astype('object')

# 3) Outliers numéricos (IQR)
outlier_summary = {}
for c in num_cols:
    Q1 = df[c].quantile(0.25)
    Q3 = df[c].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    n_out = ((df[c] < lower) | (df[c] > upper)).sum()
    outlier_summary[c] = int(n_out)
print('numeric outliers per col:', outlier_summary)

# 4) Categóricas: agrupar niveles raros (<1%) como 'rare'
rare_thresh = 0.01
cat_rare_counts = {}
for c in cat_cols:
    freqs = df[c].value_counts(normalize=True)
    rare_levels = freqs[freqs < rare_thresh].index.tolist()
    cat_rare_counts[c] = len(rare_levels)
    if rare_levels:
        df[c] = df[c].where(~df[c].isin(rare_levels), other='rare')
print('rare levels replaced per cat col (count):')
print(cat_rare_counts)

# 5) Imputación: numérica -> mediana, categórica -> moda
if num_cols:
    num_imp = SimpleImputer(strategy='median')
    df[num_cols] = num_imp.fit_transform(df[num_cols])

cat_imp = SimpleImputer(strategy='most_frequent')
if cat_cols:
    df[cat_cols] = cat_imp.fit_transform(df[cat_cols])

print('Nulls after imputation:')
print(df.isnull().sum().sum())

# 6) Resumen y guardado
print('Final shape:', df.shape)
out_path = '../clean_data/mushrooms_cleaned_advanced.csv'
df.to_csv(out_path, index=False)
print('Wrote', out_path)

duplicates: 0
After drop duplicates shape: (8124, 23)
num_cols: ['cluster']
cat_cols len: 22
numeric outliers per col: {'cluster': 0}
rare levels replaced per cat col (count):
{'class': 0, 'cap-shape': 2, 'cap-surface': 1, 'cap-color': 3, 'bruises': 0, 'odor': 1, 'gill-attachment': 0, 'gill-spacing': 0, 'gill-size': 0, 'gill-color': 2, 'stalk-shape': 0, 'stalk-root': 0, 'stalk-surface-above-ring': 1, 'stalk-surface-below-ring': 0, 'stalk-color-above-ring': 2, 'stalk-color-below-ring': 2, 'veil-color': 1, 'ring-number': 1, 'ring-type': 2, 'spore-print-color': 5, 'population': 0, 'habitat': 0}
Nulls after imputation:
0
Final shape: (8124, 23)
Wrote ../clean_data/mushrooms_cleaned_advanced.csv


## Reducción de dimensionalidad: PCA + t-SNE
Aplicamos dos técnicas complementarias:
1. **PCA (Principal Component Analysis):** Reduce preservando 95% de la varianza. Útil para entrenar modelos rápidamente con menos features.
2. **t-SNE (t-Distributed Stochastic Neighbor Embedding):** Reduce a 2D manteniendo la estructura local de los datos. Perfecto para visualización.

Proceso:
- OneHotEncode las columnas categóricas → 113 features
- StandardScaler para normalizar
- PCA con 95% varianza → ~59 componentes
- t-SNE en 2D (después de reducir PCA a 50 dims para eficiencia) → visualización
- Guardar ambas versiones en CSV para usar en modelos posteriores

In [18]:
# Identificar columnas categóricas y numéricas
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('cat_cols:', len(cat_cols), '| num_cols:', len(num_cols))

# 1) OneHotEncode + StandardScaler
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
], remainder='passthrough' if num_cols else 'drop')

X = preprocessor.fit_transform(df[cat_cols + num_cols])
print('After OneHot shape:', X.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2) PCA con 95% de varianza
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
print(f'After PCA (95% var) shape: {X_pca.shape}')
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.4f}')

# 3) t-SNE (primero reducir a 50 dims para eficiencia)
dims_for_tsne = min(50, X_pca.shape[1])
if X_pca.shape[1] > dims_for_tsne:
    pca_reduce = PCA(n_components=dims_for_tsne)
    X_pca_reduced = pca_reduce.fit_transform(X_pca)
else:
    X_pca_reduced = X_pca

print('Input to t-SNE shape:', X_pca_reduced.shape)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_pca_reduced)
print('After t-SNE shape:', X_tsne.shape)

# 4) Guardar resultados
df_pca = pd.DataFrame(X_pca, columns=[f'pca_{i}' for i in range(X_pca.shape[1])])
df_tsne = pd.DataFrame(X_tsne, columns=['tsne_1', 'tsne_2'])

# Añadir clase original si existe
if 'class' in df.columns:
    df_pca['class'] = df['class'].values
    df_tsne['class'] = df['class'].values

df_pca.to_csv('../clean_data/mushrooms_pca.csv', index=False)
df_tsne.to_csv('../clean_data/mushrooms_tsne.csv', index=False)

print(f'Saved: mushrooms_pca.csv ({df_pca.shape}), mushrooms_tsne.csv ({df_tsne.shape})')

C:\Users\jaime\AppData\Local\Temp\ipykernel_16644\669462089.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()


cat_cols: 22 | num_cols: 1
After OneHot shape: (8124, 107)
After PCA (95% var) shape: (8124, 54)
Explained variance: 0.9537
Input to t-SNE shape: (8124, 50)
After t-SNE shape: (8124, 2)
Saved: mushrooms_pca.csv ((8124, 55)), mushrooms_tsne.csv ((8124, 3))


## Selección de variables y clustering no supervisado

Esta celda identifica columnas categóricas "útiles" descartando columnas constantes, casi constantes (≥95% de la misma categoría) y con alta cardinalidad (>50 niveles).
A continuación realiza codificación (one-hot), reduce la dimensión con `TruncatedSVD` y aplica `KMeans` probando k entre 2 y 5. Se evalúa cada partición con la puntuación de silhouette y se guarda la mejor asignación en `clean_data/mushrooms_with_clusters.csv`.

Puntos a tener en cuenta:
- Si quedan menos de 2 variables útiles, se añaden columnas candidatas con cardinalidad ≤100 (hasta 5).
- Para datasets grandes preferible usar `OneHotEncoder(sparse_output=True)` y aplicar `TruncatedSVD` sobre la matriz dispersa para ahorrar memoria.
- Ajusta umbrales (`top_freq`, `n_unique`) según tus necesidades.


In [19]:
# Selección de variables útiles y clustering no supervisado
# Analizamos columnas categóricas y descartamos constantes/quasi-constantes/alta cardinalidad
cat_cols = df.select_dtypes(include=['object','string','category']).columns.tolist()
if 'class' in cat_cols:
    cat_cols.remove('class')
summary = []
drop_reasons = {}
useful = []
for c in cat_cols:
    n_unique = df[c].nunique(dropna=False)
    top_freq = df[c].value_counts(normalize=True, dropna=False).iloc[0]
    n_missing = int(df[c].isna().sum())
    reason = []
    if n_unique <= 1:
        reason.append('constant')
    if top_freq >= 0.95:
        reason.append('quasi_constant')
    if n_unique > 50:
        reason.append('high_cardinality')
    if reason:
        drop_reasons[c] = ','.join(reason)
    else:
        useful.append(c)
    summary.append({'col':c,'n_unique':int(n_unique),'top_freq':float(top_freq),'n_missing':n_missing,'keep': c in useful})
import pandas as _pd
print('Variables analizadas:', len(cat_cols))
print('Conservar (ejemplo 20):', useful[:20])
print('Descartar (ejemplo 20):', list(drop_reasons.items())[:20])
summary_df = _pd.DataFrame(summary)
display(summary_df.sort_values(['keep','n_unique'], ascending=[False,False]).head(50))
# Preparar features para clustering usando las útiles (ajusta umbrales si hace falta)
features = useful
# Si no hay suficientes features útiles, incluir columnas con alta cardinalidad controladamente
if len(features) < 2:
    candidates = [c for c in cat_cols if c not in features and df[c].nunique() <= 100][:5]
    features += candidates
    print('Añadidos candidatos por falta de features útiles:', candidates)
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
# One-hot encode (dense) - para datasets grandes usa sparse_output=True y TruncatedSVD sobre sparse matrix
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_ohe = ohe.fit_transform(df[features])
print('After OHE shape:', X_ohe.shape)
# Reducir dimensión con TruncatedSVD para acelerar clustering
n_components = min(20, max(2, X_ohe.shape[1] // 10))
svd = TruncatedSVD(n_components=n_components, random_state=0)
X_red = svd.fit_transform(X_ohe)
print('After SVD shape:', X_red.shape)
# Buscar el mejor k con silhouette entre 2 y 5
best = {'score': -1, 'n_clusters': None, 'labels': None}
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = km.fit_predict(X_red)
    try:
        score = silhouette_score(X_red, labels)
    except Exception:
        score = -1
    print(f'k={k} silhouette={score:.4f}')
    if score > best['score']:
        best.update({'score': score, 'n_clusters': k, 'labels': labels})
print('Mejor k:', best['n_clusters'], 'score:', best['score'])
df['cluster_unsupervised'] = best['labels'] if best['labels'] is not None else pd.NA
out_path = '../clean_data/mushrooms_with_clusters.csv'
df.to_csv(out_path, index=False)
print('Guardado', out_path)

Variables analizadas: 21
Conservar (ejemplo 20): ['cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']
Descartar (ejemplo 20): [('gill-attachment', 'quasi_constant'), ('veil-color', 'quasi_constant')]


,col,n_unique,top_freq,n_missing,keep
8,gill-color,11,0.212703,0,True
4,odor,9,0.434269,0,True
2,cap-color,8,0.281142,0,True
13,stalk-color-above-ring,8,0.549483,0,True
14,stalk-color-below-ring,8,0.539636,0,True
20,habitat,7,0.387494,0,True
19,population,6,0.497292,0,True
0,cap-shape,5,0.450025,0,True
18,spore-print-color,5,0.293944,0,True
1,cap-surface,4,0.399311,0,True


After OHE shape: (8124, 98)
After SVD shape: (8124, 9)
k=2 silhouette=0.2856
k=3 silhouette=0.3770
k=4 silhouette=0.4134
k=5 silhouette=0.4690
Mejor k: 5 score: 0.46896817614040026
Guardado ../clean_data/mushrooms_with_clusters.csv
